In [ ]:
import re
import spacy

nlp = spacy.load("en_core_web_sm")  # o "es_core_news_sm" si tu dataset está en español

def clean_and_lemmatize(text, leakage_terms=None):
    # 1. Quitar HTML/URLs
    text = re.sub(r'http\S+|www\S+', '', text)
    # → "BREAKING: Scientists Say the Earth is NOT Round!! Click here: "

    # 2. Minúsculas
    text = text.lower()
    # → "breaking: scientists say the earth is not round!! click here: "

    # 3. Quitar puntuación/caracteres raros
    text = re.sub(r'[^a-záéíóúñ\s]', '', text)
    # → "breaking scientists say the earth is not round click here "

    # 4. Quitar marcadores de leakage (si detectaste alguno en el EDA)
    if leakage_terms:
        for term in leakage_terms:
            text = text.replace(term.lower(), '')

    # 5. Tokenizar + quitar stopwords + lematizar (esto lo hace spaCy)
    doc = nlp(text)
    tokens = [tok.lemma_ for tok in doc if not tok.is_stop and tok.is_alpha]
    # → ["break", "scientist", "say", "earth", "round", "click"]
    # (nota: "not" desaparece por ser stopword — para TF-IDF eso está bien,
    #  a diferencia de la BiLSTM del notebook, que la mantenía)

    return " ".join(tokens)
    # → "break scientist say earth round click"

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

# --- FASE DE ENTRENAMIENTO (una sola vez, con tus datos de train) ---
textos_limpios_train = [clean_and_lemmatize(t) for t in textos_crudos_train]
# → ["break scientist say earth round click", "president sign new law today", ...]

vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))
X_train = vectorizer.fit_transform(textos_limpios_train)
# Aquí es donde el vectorizer APRENDE el vocabulario (qué 5000 palabras/bigramas
# va a usar) y calcula los pesos IDF. Esto es lo que se guarda como artifact:
joblib.dump(vectorizer, 'vectorizer_tfidf.pkl')

# X_train ya NO es texto — es una matriz dispersa de números,
# una fila por noticia, una columna por palabra del vocabulario aprendido.

In [ ]:
def preprocess_text(title, text, leakage_terms):
    # Etapa 1: texto → texto limpio (código fijo, sin estado aprendido)
    cleaned = clean_and_lemmatize(f"{title}. {text}", leakage_terms)
    return cleaned  # sigue siendo un string: "break scientist say earth round click"

# --- uso en el servicio ---
cleaned = preprocess_text(titulo_nuevo, texto_nuevo, leakage_terms)
X_nuevo = vectorizer.transform([cleaned])   # Etapa 2: texto limpio → números (usa el artifact)
prob = model.predict_proba(X_nuevo)[0, 1]   # Etapa 3: números → predicción

texto crudo (str)
   → clean_and_lemmatize()          [código fijo]
texto limpio (str)
   → vectorizer.transform()         [artifact: vectorizer_tfidf.pkl]
vector numérico (matriz dispersa)
   → model.predict_proba()          [artifact: model_clasificador.pkl]
probabilidad (float)

In [ ]:
# Artifacts entrenados en tu fase de Modelado, cargados una sola vez
def load_artifacts():
    artifacts = {}
    artifacts['vectorizer'] = joblib.load('vectorizer_tfidf.pkl')   # TF-IDF ya ajustado
    artifacts['model'] = joblib.load('model_clasificador.pkl')      # LogReg/XGBoost/etc.
    artifacts['threshold'] = json.load(open('threshold.json'))['best_threshold']
    return artifacts

def preprocess_text(title, text):
    # Tu función de limpieza (equivalente a clean_news_text del notebook BiLSTM,
    # pero con stopwords fuera y lematización, según decidiste en el paso 7)
    combined = clean_and_lemmatize(f"{title}. {text}")
    return combined

def clasificar_noticia(probability, threshold):
    # Equivalente a clasificar_riesgo(), pero fake/real en vez de niveles de riesgo
    return "Fake" if probability >= threshold else "Real"

def buscar_evidencia(title, text):
    # Aquí va tu herramienta de RAG/búsqueda — el equivalente
    # a generar_recomendaciones(), pero basado en evidencia externa, no reglas fijas
    ...
    return evidencias  # lista de fuentes/snippets relacionados

def fake_news_service(title, text):
    artifacts = load_artifacts()

    # 1. Preprocesado + vectorización
    cleaned = preprocess_text(title, text)
    X = artifacts['vectorizer'].transform([cleaned])

    # 2. Clasificación
    probability = artifacts['model'].predict_proba(X)[0, 1]
    prediccion = clasificar_noticia(probability, artifacts['threshold'])

    # 3. Evidencia externa (esto es lo que NO tiene el repo de referencia)
    evidencias = buscar_evidencia(title, text)

    return {
        "prediccion": prediccion,
        "probabilidad": round(float(probability), 4),
        "evidencias": evidencias,
    }

Si tu diferencial de TFM es un agente de verdad (el que discutimos antes: LLM que decide si llamar al clasificador, si buscar evidencia adicional vía RAG, y luego sintetiza)

tu_proyecto/
├── src/
│   └── preprocessing.py      ← aquí vive clean_and_lemmatize() UNA sola vez
├── notebooks/
│   ├── Preprocesado.ipynb    ← from src.preprocessing import clean_and_lemmatize
│   └── Modelado.ipynb
├── Artifacts/
│   ├── vectorizer_tfidf.pkl
│   ├── model_clasificador.pkl
│   └── leakage_terms.json
└── api_backend.py            ← from src.preprocessing import clean_and_lemmatize

Fíjate que esto es justo lo que hace (de forma implícita) el repo de referencia que me pasaste: api_backend.py no redefine su propia limpieza de columnas, usa el mismo io_schema.json que se generó en Modelado.ipynb. La diferencia es que ellos comparten un artifact de configuración (el schema), y tú además vas a compartir código (la función), porque tu transformación no es solo mapear columnas, es lógica de texto.

Así que la regla mental que te puedes quedar: todo lo que sea artifact (vectorizer, modelo, threshold, leakage_terms) se comparte guardándolo en disco y cargándolo; todo lo que sea código (clean_and_lemmatize) se comparte con un import, nunca duplicando el código en dos sitios.

Búsqueda web (Google/Bing/News API): evidencia factual actual — la fuente principal para "¿es esto verdad ahora mismo?"
AVeriTeC: detector de "esto suena a un tipo de bulo ya conocido" + banco de evaluación de tu agente — un complemento, no la fuente principal de evidencia en producción
Modelo ML (TF-IDF): la señal de estilo de escritura, complementaria a las dos anteriores

Extraer claims 
Aviso: esto NO usa el texto que ya limpiaste para TF-IDF

clean_and_lemmatize (sin stopwords, lematizado, todo en minúsculas) está pensado para que un vectorizador cuente palabras — pero para extraer claims necesitas que un LLM entienda la gramática y estructura de la frase ("Biden dijo que...", "según fuentes..."). Si le das el texto lematizado sin stopwords, un LLM ya no puede distinguir sujeto/verbo/objeto con fiabilidad.

Así que tu pipeline en realidad tiene dos ramas de preprocesado a partir del texto crudo, no una sola:

texto crudo (title + text)
   ├──→ clean_and_lemmatize() ──→ TF-IDF ──→ Clasificador ML
   └──→ limpieza ligera (solo HTML/URLs) ──→ extracción de claims (LLM) ──→ RAG

In [1]:
from dotenv import load_dotenv

In [2]:
load_dotenv()

True

In [2]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

In [4]:
from src.language_detector import detect_language

In [5]:
title = "La Unión Europea prohibirá completamente los pagos en efectivo en 2027"
body = "La medida entrará en vigor en 2027 y afectará a todos los países miembros."

language = detect_language(title, body)

print(language)

es


In [6]:
title = "The European Union will ban cash payments in 2027"
body = "The measure will come into force in 2027 and will affect all member states."

language = detect_language(title, body)

print(language)

en
